In [1]:
from pathlib import Path
import sys
import os
import django

BASE_DIR = Path.cwd().parent

sys.path.append(str(BASE_DIR))

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
os.environ.setdefault(
    "DJANGO_SETTINGS_MODULE",
    "config.settings"
)

django.setup()

In [3]:
import pandas as pd
import numpy as np
import products.models as mod

In [4]:
# csv 경로 읽고 데이터를 {'csv 컬럼명': 데이터} dictionary 형태로 바꿔주는 함수
def dictionarize(csv_route:str)-> list[dict]:
    rlist = []

    tar_df = pd.read_csv(csv_route).replace({np.nan: None, "": None})
    cols = tar_df.columns
    cols = [c.split("(")[0] if "(" in c else c for c in cols]
    tar_list = list(tar_df.itertuples(index=False, name=None))

    for t in tar_list:
        buff = dict()
        for i, c in enumerate(cols):
            buff[c] = t[i]
        rlist.append(buff)

    return rlist

In [5]:
# 정의된 테이블 models.Model 목록
model_list = [mod.ScreenResolution, mod.ProductAC, mod.ProductFridge, mod.ProductVAC, mod.ProductWash]
tvmod = mod.ProductTV

In [7]:
# 기존 데이터 행 모두 삭제
tvmod.objects.all().delete()

for m in model_list:
    m.objects.all().delete()

In [8]:
# 데이터 로드
for m in model_list:
    table_name = m._meta.db_table
    buff = dictionarize(f"data/database/{table_name}.csv")
    tuples = [m(**b) for b in buff]
    m.objects.bulk_create(tuples)
    print(f"{table_name} 데이터 적재 완료")

ScreenResolution 데이터 적재 완료
ProductAC 데이터 적재 완료
ProductFridge 데이터 적재 완료
ProductVAC 데이터 적재 완료
ProductWash 데이터 적재 완료


In [9]:
table_name = tvmod._meta.db_table
buff = dictionarize(f"data/database/{table_name}.csv")
for b in buff:
    b["resol_code_id"] = b["resol_code"]
    b.pop("resol_code")

tuples = [tvmod(**b) for b in buff]
tvmod.objects.bulk_create(tuples)
print(f"{table_name} 데이터 적재 완료")

ProductTV 데이터 적재 완료
